In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'kristofferc_pgfplotsx.jl',
    'fizban007_coffeegpu',
    'clayne_retrowrite',
    'dkazanc_tomophantom',
    'alicevision_meshroom',
    'fmidev_smartmet-library-newbase',
    'knausb_vcfr',
    'fedml-ai_fedml',
    'latte-central_latte',
    'stineb_mct',
]


list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00103d05f1291a1acf567cbd3f18cb60355096de,kristofferc_pgfplotsx.jl


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  1%|█                                                                             | 50/3606 [00:51<1:00:44,  1.02s/it]

Got Errors {'7342aa2ba051597afdfd2863b474012ea36f8f1f': 'Key 7342aa2ba051597afdfd2863b474012ea36f8f1f not found in /da5_fast/All.sha1c/commit_115.tch'}


  2%|█▎                                                                            | 61/3606 [01:02<1:00:34,  1.03s/it]

Got Errors {'86cd1e530fc4a311c0679b186facec1e49fa64e5': 'Key 86cd1e530fc4a311c0679b186facec1e49fa64e5 not found in /da5_fast/All.sha1c/commit_6.tch'}


  2%|█▋                                                                            | 76/3606 [01:18<1:00:15,  1.02s/it]

...

Got Errors {'ef12ac402527298e2b3e7e44131471865eede4e2': 'Key ef12ac402527298e2b3e7e44131471865eede4e2 not found in /da5_fast/All.sha1c/commit_111.tch'}


 98%|██████████████████████████████████████████████████████████████████████████▍ | 3532/3606 [1:00:25<01:16,  1.04s/it]

Got Errors {'432d2aa26ae6ff16d7e7b52b8cd9038ce82b13fc': 'Key 432d2aa26ae6ff16d7e7b52b8cd9038ce82b13fc not found in /da5_fast/All.sha1c/commit_67.tch'}


100%|████████████████████████████████████████████████████████████████████████████| 3606/3606 [1:01:41<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00103d05f1291a1acf567cbd3f18cb60355096de,933ca1225f801eccb0ec7d301790399666f6502a,[a580cd78da6fc631d56f4f1158e9df4a05128b09],autodocs <autodocs>,1528883833,+0000,autodocs <autodocs>,1528883833,+0000,build based on 07d0d25\n,00103d05f1291a1acf567cbd3f18cb60355096de,kristofferc_pgfplotsx.jl
1,00205d9769d1eebf71c061df26c48c30badfbd8a,fb61a323692ce355647d468d1951032cf77c6e0e,[e33d254b52f8ebb15e8a779add4b6a2b64f5f4b7],Kristoffer Carlsson <kcarlsson89@gmail.com>,1534505613,+0200,Kristoffer Carlsson <kristoffer.carlsson@chalm...,1534508541,+0200,fixes for the display system\n,00205d9769d1eebf71c061df26c48c30badfbd8a,kristofferc_pgfplotsx.jl


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00103d05f1291a1acf567cbd3f18cb60355096de', 'tree': '933ca1225f801eccb0ec7d301790399666f6502a', 'parent': ['a580cd78da6fc631d56f4f1158e9df4a05128b09'], 'author': 'autodocs <autodocs>', 'author_time': 1528883833, 'author_tz': '+0000', 'committer': 'autodocs <autodocs>', 'committer_time': 1528883833, 'committer_tz': '+0000', 'message': 'build based on 07d0d25\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,stineb_mct,00103d05f1291a1acf567cbd3f18cb60355096de,autodocs <autodocs>,1528883833,build based on 07d0d25\n


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='wch356'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github